In [1]:
!uv pip install torch torchvision pillow tqdm scikit-learn streamlit opencv-python-headless numpy kagglehub

Using Python 3.12.12 environment at: /usr
Resolved 53 packages in 536ms
Prepared 2 packages in 429ms
Installed 2 packages in 17ms
 + pydeck==0.9.1
 + streamlit==1.55.0


In [2]:
import kagglehub
fia_root = kagglehub.dataset_download('nikita2998/find-it-again-dataset')
fia_root

Using Colab cache for faster access to the 'find-it-again-dataset' dataset.


'/kaggle/input/find-it-again-dataset'

In [3]:
import ast, pickle, csv
import numpy as np
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
import os
import io
import torch
from torchvision import models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

DATA_DIR  = Path(os.path.join(fia_root, "findit2"))
MODEL_OUT = "forgery_model.pkl"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
backbone.classifier = torch.nn.Identity()
backbone = backbone.to(DEVICE).eval()

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def compute_ela(img, quality=75, amplify=10):
    # re-save at lower quality and compute difference so tampered regions show brighter
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    recompressed = Image.open(buffer).convert("RGB")
    ela = ImageChops.difference(img, recompressed)
    ela = ImageEnhance.Brightness(ela).enhance(amplify)
    return ela

def load_split(txt_file):
    split = txt_file.replace(".txt", "")
    paths, labels, bboxes_list = [], [], []

    with open(DATA_DIR / txt_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # print("row is", row)
            p = DATA_DIR / split / row["image"].strip()
            if not p.exists():
                continue
            paths.append(p)
            labels.append(int(row["forged"]))

            boxes = []
            ann = row["forgery annotations"].strip()
            if int(row["forged"]) == 1 and ann not in ("0", ""):
                try:
                    d = ast.literal_eval(ann)
                    for reg in d.get("regions", []):
                        sa, ra = reg["shape_attributes"], reg["region_attributes"]
                        if str(ra.get("Original area", "no")).lower() == "yes":
                            continue
                        if sa.get("name") == "rect":
                            boxes.append([sa["x"], sa["y"], sa["width"], sa["height"]])
                except Exception:
                    pass
            bboxes_list.append(boxes)

    print(f"  {txt_file}: {len(paths)} imgs | forged={sum(labels)}")
    return paths, labels, bboxes_list

def extract_features(paths):
    feats = []
    for p in paths:
        img = Image.open(p).convert("RGB")

        # original image features
        orig_feat = backbone(tf(img).unsqueeze(0).to(DEVICE)).squeeze().cpu().detach().numpy()

        # ELA image features highlightingtampered regions
        ela_feat = backbone(tf(compute_ela(img)).unsqueeze(0).to(DEVICE)).squeeze().cpu().detach().numpy()

        # concatenating boht
        feats.append(np.concatenate([orig_feat, ela_feat]))

    return np.array(feats)


tr_paths, tr_labels, tr_bboxes = load_split("train.txt")
va_paths, va_labels, _ = load_split("val.txt")
te_paths, te_labels, _ = load_split("test.txt")
print("Extracting features (train)...")
X_train = extract_features(tr_paths)
print("Extracting features (val+test)...")
X_val   = extract_features(va_paths)
X_test  = extract_features(te_paths)
print("Training LogisticRegression...")
clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
clf.fit(X_train, tr_labels)
print(f"Val accuracy : {clf.score(X_val, va_labels):.3f}")
print("\n\n\nTest Report:")
print(classification_report(te_labels, clf.predict(X_test),
                             target_names=["authentic", "forged"]))
bbox_lookup = {p.name: b for p, b in zip(tr_paths, tr_bboxes)}
with open(MODEL_OUT, "wb") as f:
    pickle.dump({"clf": clf, "bbox_lookup": bbox_lookup}, f)
print(f"Saved to {MODEL_OUT}")

Device: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 112MB/s] 


  train.txt: 577 imgs | forged=94
  val.txt: 192 imgs | forged=33
  test.txt: 218 imgs | forged=35
Extracting features (train)...
Extracting features (val+test)...
Training LogisticRegression...
Val accuracy : 0.641



Test Report:
              precision    recall  f1-score   support

   authentic       0.88      0.81      0.84       183
      forged       0.29      0.40      0.33        35

    accuracy                           0.74       218
   macro avg       0.58      0.60      0.59       218
weighted avg       0.78      0.74      0.76       218

Saved to forgery_model.pkl
